In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/cleaned_churn.csv")

In [2]:
df_fe = df.copy()

In [3]:
## dropping irrelevant columns

cols_to_drop = [
    "CustomerID",
    "Lat Long",
    "Latitude",
    "Longitude",
    "Churn Label",
    "Churn Score",
    "Churn Reason"
]

df_fe.drop(columns=cols_to_drop, inplace=True)

In [4]:
## sevices per customer

service_columns = [
    "Phone Service",
    "Multiple Lines",
    "Internet Service",
    "Online Security",
    "Online Backup",
    "Device Protection",
    "Tech Support",
    "Streaming TV",
    "Streaming Movies"
]

df_fe["Service Count"] = (
    df_fe[service_columns]
    .isin(["Yes", "DSL", "Fiber optic"])
    .sum(axis=1)
)

In [5]:
# monthly charge category
df_fe["Charge Category"] = pd.cut(
    df_fe["Monthly Charges"],
    bins=[0, 35, 70, 120],
    labels=["Low", "Medium", "High"]
)

In [6]:
# tenure groups

df_fe["Tenure Group"] = pd.cut(
    df_fe["Tenure Months"],
    bins=[0, 12, 24, 48, 72],
    labels=[
        "New",
        "Regular",
        "Loyal",
        "Very Loyal"
    ]
)

In [8]:
#avg monthly spend check

df_fe["Avg Monthly Spend"] = (
    df_fe["Total Charges"] /
    df_fe["Tenure Months"]
)

df_fe["Avg Monthly Spend"] = (
    df_fe["Avg Monthly Spend"]
    .fillna(df_fe["Monthly Charges"])
)

In [9]:
## high value customer
df_fe["High Value Customer"] = (
    df_fe["CLTV"] >
    df_fe["CLTV"].median()
).astype(int)

In [12]:
## family customer

df_fe["Family Customer"] = (
    (df_fe["Partner"] == "Yes") &
    (df_fe["Dependents"] == "Yes")
).astype(int)

In [13]:
## long term customer

df_fe["Long Term Customer"] = (
    df_fe["Tenure Months"] >= 24
).astype(int)

In [14]:
## internet user

df_fe["Internet User"] = (
    df_fe["Internet Service"] != "No"
).astype(int)

In [15]:
df_fe.head()

,Count,Country,State,City,Zip Code,Gender,Senior Citizen,Partner,Dependents,Tenure Months,...,Churn Value,CLTV,Service Count,Charge Category,Tenure Group,Avg Monthly Spend,High Value Customer,Family Customer,Long Term Customer,Internet User
0,1,United States,California,Los Angeles,90003,Male,No,No,No,2,...,1,3239,4,Medium,New,54.075000,0,0,0,1
1,1,United States,California,Los Angeles,90005,Female,No,No,Yes,2,...,1,2701,2,High,New,75.825000,0,0,0,1
2,1,United States,California,Los Angeles,90006,Female,No,No,Yes,8,...,1,5372,6,High,New,102.562500,1,0,0,1
3,1,United States,California,Los Angeles,90010,Female,No,Yes,Yes,28,...,1,5003,7,High,Loyal,108.787500,1,1,1,1
4,1,United States,California,Los Angeles,90015,Male,No,No,Yes,49,...,1,5340,7,High,Very Loyal,102.781633,1,0,1,1


In [17]:
df.columns

Index(['CustomerID', 'Count', 'Country', 'State', 'City', 'Zip Code',
       'Lat Long', 'Latitude', 'Longitude', 'Gender', 'Senior Citizen',
       'Partner', 'Dependents', 'Tenure Months', 'Phone Service',
       'Multiple Lines', 'Internet Service', 'Online Security',
       'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV',
       'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method',
       'Monthly Charges', 'Total Charges', 'Churn Label', 'Churn Value',
       'Churn Score', 'CLTV', 'Churn Reason'],
      dtype='str')

In [18]:
## saving
df_fe.to_csv(
    "../data/processed/feature_engineered_data.csv",
    index=False
)